# MF-map variant (Kaggle T4): D4 TTA of the Mask2Former maps

Optional step after `reproduce_maps.ipynb`, on a Kaggle **T4** runtime. No training. It writes, for each MF fold model (MF0, MF1, MF2) and the 180 test images, one more archive next to the package maps:

- `MF{k}_tta.npz`: the mean of the 8 D4 views (rotations by 0/90/180/270 degrees, with and without a horizontal flip) of the 1024-px MF input, each output transformed back before averaging at the network's 256-px output.

`--views tta tiled` also writes `MF{k}_tiled.npz` (the native 2048-px image read as nine 1024-px tiles at stride 512, the tile foregrounds Hann-blended), which the `mfhires` and `mfcomb` modes read; those variants were not adopted and are not run here.

The maps are upsampled to 2048 px and stored like the package maps (uint16, value / 510). The code is `filseg.reproduction.mfmaps`; the view geometry is `mf_d4.py` and `mf_tile.py`, byte-identical copies of the cross-validated lanes' code. The package maps (`full`, `local`, `MF0`, `MF1`, `MF2`, map set `d7aa6a41fafc26b07ce89b306b884f6667ebb68b86289ea4745ee44c0b0d195f`) are read, never rewritten.

**Package maps.** Run this notebook in the same session as `reproduce_maps.ipynb` (same `OUTPUT`), or attach that notebook's output: its five archives are then copied into `OUTPUT`. With the package maps present, the identity view of every image is checked against `MF{k}.npz` and the map-set sha256 of each variant is recorded in `mf_views_result.json`.

**Deterministic mode** is the one of `reproduce_maps.ipynb` (on by default; `--nondeterministic` switches it off). MF uses its original BF16 dtype.

**Expected result.** The `mftta` map set (full, local, MF0_tta, MF1_tta, MF2_tta) has the sha256 `44ee26c7d4284b02103f005935611b0911d8f1fe20e1693bf73f0eb21fff9cc2` (sha256 of the JSON of the archive sha256, sorted keys). With `--views tta tiled`: `mfhires` `6f97d421381f6470ff7748726bad27fcc89fde0c74fdf5dc2f9610125e54255e`, `mfcomb` `969adfb0f90fce9589424936423a64edfb8e7715dd191e9e779ac688254cbda1`.

The CSV stage reads them in the `"mftta"` mode: `reproduce_submission.ipynb` and `pipeline.ipynb` find this output by itself (this session's `/kaggle/working/maps` first, then one attached output); from the command line, `python -m filseg.reproduction csv --mode mftta --maps <package maps> --variant-maps OUTPUT`.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, shutil
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Clone this repository at its release tag (needs Internet), unless this notebook already runs inside a checkout.
SOURCE_TAG = "v1.1"
if not (REPO / "src/filseg/reproduction").is_dir():
    REPO = Path("/kaggle/working/filament-reproduction-source")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Matapanino/filament-segmentation-2026-solution.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_TAG], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))  # the editable install is seen only by new processes; the next cell imports filseg here

In [ ]:
from filseg.reproduction.paths import find_test_images
IMAGE_ROOT = find_test_images()  # or set IMAGE_ROOT = Path("<the test JPEG directory>")
OUTPUT = Path("/kaggle/working/maps")
OUTPUT.mkdir(parents=True, exist_ok=True)
if not (OUTPUT / "inference_result.json").exists():
    # The package maps of an attached reproduce_maps.ipynb output, if any (read only; copied next to the variants).
    attached = [p for p in Path("/kaggle/input").rglob("inference_result.json")]
    if len(attached) == 1:
        for name in ("inference_result.json", "full.npz", "local.npz", "MF0.npz", "MF1.npz", "MF2.npz"):
            shutil.copy(attached[0].parent / name, OUTPUT / name)
subprocess.run([sys.executable, "-m", "filseg.reproduction", "infer-mfviews", "--images", str(IMAGE_ROOT), "--output", str(OUTPUT), "--views", "tta"], check=True)
result = json.loads((OUTPUT / "mf_views_result.json").read_text())
print({k: result[k] for k in ("status", "images", "elapsed_seconds", "deterministic")})
print({name: row["sha256"] for name, row in result["networks"].items()})
print({name: (row["equal_images"], row["images"]) for name, row in result["identity_check"].items()})
print(result.get("map_sets"))
